In [ ]:
import json
import datasets
from transformers import T5ForConditionalGeneration, T5Tokenizer

Due to our discorvery of 'Cantonese only characters' appearing in most sentences in SmolSent, we wondered how the translations would perform if we just did direct character replacements from Cantonese to Mandarin characters using the gatitos token level translations. 

Since gatitos also had more than one character we also explore how the translations perform if we to a replacing of more characters.
An example where this was relevant is '椰菜' which refers to cabbage, but literally means ["coconut vegetable"](https://en.wiktionary.org/wiki/%E6%A4%B0%E8%8F%9C). So only doing 1 character replacements would result in this being translated to coconut, but doing multiple character replacements would result in this being translated correctly to cabbage.

In [ ]:
yue_zh_characters_validation = datasets.load_dataset("google/smol", "gatitos__yue_zh")
with open("character_classification.json", "r") as f:
    character_classfication = json.load(f)
    
# The 19 worst MADLAD400-3b SmolSent en_yue translations (ranked by BLEURT)
with open("bad_translations.json", "r", encoding="utf-8") as f:
    bad_translations = json.load(f)

In [6]:
yue_zh_characters_validation = datasets.load_dataset("google/smol", "gatitos__yue_zh")

This is the function use to replace single characters

In [ ]:
def replace_canto_only_characters(s: str) -> None | str:
    s_out = []
    for c in s:
        is_canto_only = c in character_classfication and character_classfication[c] == 'CANTO_ONLY'
        is_in_gatitos = c in yue_zh_characters_validation['train']['src']
        if is_canto_only and is_in_gatitos: # type: ignore
            token_translations = [x for x in yue_zh_characters_validation['train'] if x['src'] == c] # type: ignore
            s_out.append(token_translations[0]['trgs'][-1]) # type: ignore
        else:
            s_out.append(c)
            
    return ''.join(s_out)

This is the function used to generate complete gatitos token replacements.

In [ ]:
characters = yue_zh_characters_validation["train"].to_pandas().sort_values(by="src", key=lambda x:x.str.len(), ascending=False)

def swap_tokens(sentence):
    result = ""
    i = 0
    while i < len(sentence):
        found = False
        for k in range(len(characters)):
            if characters["src"][k] == sentence[i:i+len(characters["src"][k] )]:
                result = result + characters["trgs"][k][0].split(";")[0]
                i += len(characters["src"][k])
                found = True
                break
        if not found:
            result += sentence[i]
            i += 1
    
    return result


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 237bd03d-7482-48e0-bfab-40a1e3442836)')' thrown while requesting HEAD https://huggingface.co/jbochi/madlad400-3b-mt/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


In [ ]:
model_name = 'jbochi/madlad400-3b-mt'
model = T5ForConditionalGeneration.from_pretrained(model_name, device_map=None)
tokenizer = T5Tokenizer.from_pretrained(model_name)

results = []

def translate(cantonese: str) -> str:
    text = f"<2en> {cantonese}"
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    outputs = model.generate(input_ids=input_ids, max_new_tokens=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

for item in bad_translations:
    translation_plain = translate(item['original_yue'])
    translation_swapped_tokens = translate(swap_tokens(item['original_yue']))
    translation_replaced_chars = translate(replace_canto_only_characters(item['original_yue'])) # type: ignore

    record = {
        "original": item['original'], # This is the original english sentence
        "cantonese": translation_plain, # This is the english sentence translated from Cantonese
        "tkn-swaps": translation_swapped_tokens, # This is the english sentence translated from Cantonese with multiple character token replacements
        "chr-swaps": translation_replaced_chars, # This is the english sentence translated from Cantonese with single character token replacements
    }

    results.append(record)

with open('sentence_translations_madlad_token_replacements.json', 'w+') as f:
    json.dump(results, f, indent=4)


We manually inspected the resulting translations and found some of the translations had improved significantly for example the following:
```json
    {
        "original": "Rohan was my sibling who always flew a kite at noon, even if it was cloudy.",
        "cantonese": "The bridge is opened to the public at noon, and closed at midnight.",
        "tkn-swaps": "Rohan is my brother, and he flies kites at noon every day, even when the sky is dark.",
        "chr-swaps": "Rohan is my brother, and he flies kites every day at noon, even in the dark."
    }
```
Here we see the version translated directly from Cantonese bears little to no resemblance to the original English version. But the versions using gatitos replacements do.

This example shows how doing multiple character gatitos replacements can improve results over using single character replacements:
```json
    {
        "original": "Not to mention democrats are already split over the infrastructure bill, lmao.",
        "cantonese": "The government has not yet made any official statement on the issue, but the opposition has been calling for a referendum on the issue, which is being held in the country.",
        "tkn-swaps": "I don't think the Democrats themselves have a different opinion on that infrastructure bill, haha.",
        "chr-swaps": "The government has not yet made any official statement on the issue, but the opposition has been calling for a referendum on the issue, which has been delayed."
    }
```

Some of the sentences did not improve, mostly because they were already decent translations (according to our manual inspection).
Of the 19 sentences we consider gatitos replacements to improve the translation quality of 11 sentences. Of the remaining 8 sentences, 6 were already good, and 2 did not get better.